# 02 - LLM Triple Extraction (ContraDoc)

For each document in ContraDoc, send the full text to an LLM in one structured-output call. The model splits the document into sentences and extracts claim triples per sentence; each triple is tagged with its 1-indexed `sentence_id` and verbatim `source_text` so downstream NLI (notebook 06) can read the original sentence rather than a verbalized triple.

**Input:** `data/processed/ContraDoc/ContraDoc.csv`

**Output:** `data/processed/ContraDoc/triples.jsonl` — one JSON object per document, schema documented below.

**Downstream consumers**
- `03_insert_to_neo4j_ContraDoc.ipynb` — flattens to per-triple records and inserts into Neo4j; each triple node carries `(doc_id, sentence_id, source_text)` as properties.
- `06_NLI_ContraDoc.ipynb` — at judgment time, looks up `source_text` for the candidate triples and uses those as the (premise, hypothesis) pair (not the verbalized triple).

In [1]:
import json
from pathlib import Path

import pandas as pd
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from tqdm import tqdm

from config import settings

INPUT_PATH = Path("data/processed/ContraDoc/ContraDoc.csv")
OUTPUT_PATH = Path("data/processed/ContraDoc/triples.jsonl")

## Structured-output schema

The LLM returns a list of sentences in document order, each with its 1-indexed `sentence_id`, verbatim `source_text`, and a (possibly empty) list of `(s, p, o)` triples extracted from that sentence.

In [2]:
class Triple(BaseModel):
    s: str = Field(description="Subject — noun phrase, with pronouns and bare definites resolved to their antecedent using document context.")
    p: str = Field(
        description="Predicate — short verbal phrase. Preserve negation ('did not donate', not 'donate') and modal/factive verbs ('managed to enter', 'may resign', 'forgot to take')."
    )
    o: str = Field(description="Object — noun phrase, numeric value, date, or short complement.")


class SentenceExtraction(BaseModel):
    sentence_id: int = Field(description="1-indexed sentence position within the document, in original order.")
    source_text: str = Field(
        description="Verbatim sentence text, exactly as it appears in the document. Do not paraphrase, normalize, or trim."
    )
    triples: list[Triple] = Field(
        description="All claim triples extracted from this sentence. Empty list if the sentence has no extractable claim."
    )
    is_evidence: bool = Field(
        default=False,
        description="True on the ONE sentence matching the Evidence metadata (YES docs only). False for all other sentences and for every sentence in NO docs.",
    )
    ref_index: int | None = Field(
        default=None,
        description="0-based index into the Reference sentences list, set on the sentence matching that Reference (YES docs only). None otherwise.",
    )


class DocumentExtraction(BaseModel):
    sentences: list[SentenceExtraction]

## LLM client and prompt

Model and API key both pulled from `config.settings`. Change `llm_model` in `config.py` to swap models without touching this notebook.

In [3]:
SYSTEM_PROMPT = """You are an information extractor. Given a document, split it into sentences and extract all claim triples (subject, predicate, object) per sentence.

Rules for extraction:
- Resolve pronouns and bare definites to their antecedents using surrounding context (e.g., 'she' -> 'Mrs. Tittlemouse'; 'the company' -> 'Microsoft Israel').
- Predicates MUST preserve polarity ('did not donate', not 'donate') and modal/factive verbs ('managed to escape', 'may resign', 'forgot to take').
- Subjects and objects should be noun phrases, numeric values, or dates — concise but specific.
- A sentence with no extractable claim has an empty `triples` list, but the sentence MUST still appear in the output (with its source_text) so sentence_id stays aligned with the document.
- Preserve sentences in document order. Do not split, merge, paraphrase, or omit them.
- `source_text` must match the document character-for-character (including punctuation and casing).

Rules for contradiction metadata (only when the user message includes an Evidence/Reference block):
- Set `is_evidence=true` on EXACTLY ONE output sentence — the one matching the provided Evidence text. All others MUST have `is_evidence=false`.
- For each Reference [i] in the provided list, set `ref_index=i` on the ONE output sentence matching that Reference text. All other sentences MUST have `ref_index=null`.
- Sentences tagged `is_evidence=true` or `ref_index=i` MUST each carry at least one triple — these are the semantically important sentences.
- If the user message does NOT include a contradiction metadata block, every sentence must have `is_evidence=false` and `ref_index=null`.
"""


llm = ChatOpenAI(model=settings.llm_model, api_key=settings.openai_api_key, temperature=0)
extractor = llm.with_structured_output(DocumentExtraction)


def extract_document(text: str, evidence: str | None = None, refs: list[str] | None = None) -> DocumentExtraction:
    user_msg = f"Document:\n{text}"
    if evidence is not None:
        user_msg += (
            "\n\n--- Contradiction metadata ---"
            "\nEvidence sentence (tag the matching output sentence with is_evidence=true; extract at least one triple from it):"
            f"\n{evidence}"
        )
    if refs:
        user_msg += (
            "\n\nReference sentence(s) that the Evidence contradicts "
            "(tag each matching output sentence with the correct ref_index; extract at least one triple from each):"
        )
        for i, ref in enumerate(refs):
            user_msg += f"\n  [{i}] {ref}"
    return extractor.invoke(
        [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_msg},
        ]
    )

## Load ContraDoc

In [4]:
contra_df = pd.read_csv(INPUT_PATH)
print(f"Loaded {len(contra_df)} documents from {INPUT_PATH}")
print(f"  YES: {(contra_df['contradiction'] == 'YES').sum()}  NO: {(contra_df['contradiction'] == 'NO').sum()}")

# Skip YES docs with no ref_sentences (no ground-truth contradiction pair recoverable).
# Keep all NO docs (their ref_sentences is 'none' by definition).
yes_with_ref = contra_df[(contra_df["contradiction"] == "YES") & (contra_df["ref_sentences"] != "none")]
no_docs = contra_df[contra_df["contradiction"] == "NO"]

SAMPLE_SIZE = 50
SEED = 42
contra_df = pd.concat(
    [
        yes_with_ref.sample(n=SAMPLE_SIZE, random_state=SEED),
        no_docs.sample(n=SAMPLE_SIZE, random_state=SEED),
    ],
    ignore_index=True,
)

print(f"Sample: YES={SAMPLE_SIZE} (with ref_sentences), NO={SAMPLE_SIZE}, total={len(contra_df)}")
contra_df.head(2)

Loaded 891 documents from data\processed\ContraDoc\ContraDoc.csv
  YES: 449  NO: 442
Sample: YES=50 (with ref_sentences), NO=50, total=100


,id,contradiction,doc_type,scope,contra_plug,contra_type,evidence,ref_sentences,text
0,3499318673_1,YES,wiki,global,Insert,Content|Numeric,Little Gidding is the first poem of T. S. Elio...,= Little Gidding ( poem ) = Little Gidding ...,= Little Gidding ( poem ) =\n\nLittle Gidding ...
1,3489738292_5,YES,news,global,Replace,Numeric|Negation,The deal currently on the table would not prov...,"For Khamenei, the ""framework"" announced last w...",The outlines of a nuclear deal with Iran are i...


## Sanity check on a single document

Run extraction on the first document to confirm the prompt and schema behave before launching the full pass.

In [5]:
sample = contra_df[contra_df["contradiction"] == "YES"].iloc[0]
refs = sample["ref_sentences"].split("|")
result = extract_document(sample["text"], evidence=sample["evidence"], refs=refs)

print(f"doc_id={sample['id']}  contradiction={sample['contradiction']}")
print(f"sentences extracted: {len(result.sentences)}")
print(f"total triples:      {sum(len(s.triples) for s in result.sentences)}")

ev = next((s for s in result.sentences if s.is_evidence), None)
ref_ids_by_idx = {s.ref_index: s.sentence_id for s in result.sentences if s.ref_index is not None}
print(f"tagged is_evidence  sentence_id: {ev.sentence_id if ev else None}")
print(f"tagged ref sentence_ids by index: {ref_ids_by_idx}")
print()
for s in result.sentences[:5]:
    tag = "EVIDENCE" if s.is_evidence else (f"REF[{s.ref_index}]" if s.ref_index is not None else "        ")
    print(f"[{s.sentence_id}] {tag}  {s.source_text[:120]}")
    for t in s.triples:
        print(f"            -> ({t.s}, {t.p}, {t.o})")

doc_id=3499318673_1  contradiction=YES
sentences extracted: 71
total triples:      122
tagged is_evidence  sentence_id: 46
tagged ref sentence_ids by index: {0: 2}

[1]           = Little Gidding ( poem ) =
[2] REF[0]  Little Gidding is the fourth and final poem of T. S. Eliot 's Four Quartets , a series of poems that discuss time , pers
            -> (Little Gidding, is, the fourth and final poem of T. S. Eliot 's Four Quartets)
            -> (T. S. Eliot 's Four Quartets, discuss, time, perspective, humanity, and salvation)
[3]           It was first published in September 1942 after being delayed for over a year because of the air - raids on Great Britain
            -> (Little Gidding, was first published, September 1942)
            -> (Little Gidding, was delayed for over a year because of, the air-raids on Great Britain during World War II and Eliot 's declining health)
[4]           The title refers to a small Anglican community in Huntingdonshire , established by Nicholas Fe

In [6]:
result.sentences

[SentenceExtraction(sentence_id=1, source_text='= Little Gidding ( poem ) =', triples=[], is_evidence=False, ref_index=None),
 SentenceExtraction(sentence_id=2, source_text="Little Gidding is the fourth and final poem of T. S. Eliot 's Four Quartets , a series of poems that discuss time , perspective , humanity , and salvation.", triples=[Triple(s='Little Gidding', p='is', o="the fourth and final poem of T. S. Eliot 's Four Quartets"), Triple(s="T. S. Eliot 's Four Quartets", p='discuss', o='time, perspective, humanity, and salvation')], is_evidence=False, ref_index=0),
 SentenceExtraction(sentence_id=3, source_text="It was first published in September 1942 after being delayed for over a year because of the air - raids on Great Britain during World War II and Eliot 's declining health.", triples=[Triple(s='Little Gidding', p='was first published', o='September 1942'), Triple(s='Little Gidding', p='was delayed for over a year because of', o="the air-raids on Great Britain during World W

In [7]:
sample["evidence"]

"Little Gidding is the first poem of T. S. Eliot 's Four Quartets."

In [8]:
sample["contra_type"]

'Content|Numeric'

In [9]:
sample

id                                                    3499318673_1
contradiction                                                  YES
doc_type                                                      wiki
scope                                                       global
contra_plug                                                 Insert
contra_type                                        Content|Numeric
evidence         Little Gidding is the first poem of T. S. Elio...
ref_sentences     = Little Gidding ( poem ) =   Little Gidding ...
text             = Little Gidding ( poem ) =\n\nLittle Gidding ...
Name: 0, dtype: str

## Run extraction on the full corpus

Resumable: reads any existing `triples.jsonl` and skips documents already done. Output is appended one JSON object per document, flushed every row, so an interrupted run can resume mid-corpus.

Failed extractions are logged and skipped; rerunning the cell will retry them.

In [10]:
import re
from difflib import SequenceMatcher


def _normalize(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip().lower().rstrip(".!?\"'")


def _similarity(a: str, b: str) -> float:
    return SequenceMatcher(None, _normalize(a), _normalize(b)).ratio()


def fuzzy_match_sentence(target: str, sentences, threshold: float = 0.85) -> tuple[int | None, float]:
    """Find sentence_id whose source_text best matches `target`. Returns (sentence_id or None, score).
    Exact-normalized match returns score 1.0; otherwise highest SequenceMatcher ratio (>= threshold)."""
    target_norm = _normalize(target)
    for s in sentences:
        if _normalize(s.source_text) == target_norm:
            return s.sentence_id, 1.0
    best_id, best_score = None, 0.0
    for s in sentences:
        score = SequenceMatcher(None, _normalize(s.source_text), target_norm).ratio()
        if score > best_score:
            best_id, best_score = s.sentence_id, score
    return (best_id if best_score >= threshold else None), best_score


def resolve_gold_sentence(target_text: str, tagged_sentence, sentences) -> tuple[int | None, str]:
    """Two-stage resolution:
       1. Prefer the LLM's tagged sentence if its source_text is similar enough to the target.
       2. Otherwise fall back to fuzzy matching the target text against all sentences.
    Returns (sentence_id or None, resolution_tag) where resolution_tag explains which branch fired."""
    if tagged_sentence is not None:
        sim = _similarity(tagged_sentence.source_text, target_text)
        if sim >= 0.85:
            return tagged_sentence.sentence_id, "llm_tag"
    sid, score = fuzzy_match_sentence(target_text, sentences)
    if sid is not None:
        tag = "fuzzy_recovered" if tagged_sentence is None else "fuzzy_override_bad_tag"
        return sid, tag
    return None, "unmatched"


OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

done_ids: set[str] = set()
if OUTPUT_PATH.exists():
    with OUTPUT_PATH.open(encoding="utf-8") as f:
        for line in f:
            done_ids.add(json.loads(line)["doc_id"])
    print(f"Resuming: {len(done_ids)} docs already extracted")

todo = contra_df[~contra_df["id"].astype(str).isin(done_ids)].reset_index(drop=True)
print(f"To extract: {len(todo)} docs")

with OUTPUT_PATH.open("a", encoding="utf-8") as f:
    for row in tqdm(todo.itertuples(index=False), total=len(todo)):
        try:
            if row.contradiction == "YES":
                refs = row.ref_sentences.split("|")
                result = extract_document(row.text, evidence=row.evidence, refs=refs)
            else:
                refs = []
                result = extract_document(row.text)
        except Exception as exc:
            print(f"FAILED doc_id={row.id}: {type(exc).__name__}: {exc}")
            continue

        gold_evidence_id = None
        gold_ref_ids: list[int] = []
        resolution_notes: list[str] = []

        if row.contradiction == "YES":
            ev_tagged = next((s for s in result.sentences if s.is_evidence), None)
            gold_evidence_id, ev_res = resolve_gold_sentence(row.evidence, ev_tagged, result.sentences)
            if ev_res != "llm_tag":
                resolution_notes.append(f"evidence={ev_res}")

            ref_by_index = {s.ref_index: s for s in result.sentences if s.ref_index is not None}
            for i, ref_text in enumerate(refs):
                sid, ref_res = resolve_gold_sentence(ref_text, ref_by_index.get(i), result.sentences)
                if sid is not None and sid not in gold_ref_ids:
                    gold_ref_ids.append(sid)
                if ref_res != "llm_tag":
                    resolution_notes.append(f"ref[{i}]={ref_res}")

            if resolution_notes:
                print(f"INFO doc_id={row.id}: {', '.join(resolution_notes)}")
            if gold_evidence_id is None:
                print(f"WARN doc_id={row.id}: evidence unmatched (not in LLM output even by fuzzy)")
            if not gold_ref_ids:
                print(f"WARN doc_id={row.id}: no refs matched")

        record = {
            "doc_id": str(row.id),
            "contradiction": row.contradiction,
            "doc_type": row.doc_type,
            "scope": row.scope,
            "contra_plug": row.contra_plug,
            "contra_type": row.contra_type,
            "evidence": row.evidence,
            "ref_sentences": row.ref_sentences,
            "gold_evidence_sentence_id": gold_evidence_id,
            "gold_ref_sentence_ids": gold_ref_ids,
            "sentences": [s.model_dump() for s in result.sentences],
        }
        f.write(json.dumps(record, ensure_ascii=False) + "\n")
        f.flush()

print(f"Done. Output: {OUTPUT_PATH.resolve()}")

To extract: 100 docs


  2%|▏         | 2/100 [00:44<34:46, 21.29s/it]

INFO doc_id=3489738292_5: ref[0]=unmatched


  4%|▍         | 4/100 [01:22<31:36, 19.75s/it]

INFO doc_id=3499318672_2: evidence=fuzzy_override_bad_tag


...

INFO doc_id=3489738288_1: ref[1]=unmatched


 50%|█████     | 50/100 [13:19<14:42, 17.65s/it]

INFO doc_id=3488771891_7: evidence=unmatched, ref[0]=unmatched
WARN doc_id=3488771891_7: evidence unmatched (not in LLM output even by fuzzy)
WARN doc_id=3488771891_7: no refs matched


100%|██████████| 100/100 [27:18<00:00, 16.39s/it]

Done. Output: D:\AT82.05-Claim-Contradiction-Over-Knowledge-Graphs\experiments\data\processed\ContraDoc\triples.jsonl


## Verify output

In [11]:
records = [json.loads(line) for line in OUTPUT_PATH.open(encoding="utf-8")]
n_sentences = sum(len(r["sentences"]) for r in records)
n_triples = sum(sum(len(s["triples"]) for s in r["sentences"]) for r in records)

print(f"Documents extracted: {len(records)} / {len(contra_df)}")
print(f"Total sentences:    {n_sentences}")
print(f"Total triples:      {n_triples}")
print(f"Avg triples / doc:  {n_triples / max(len(records), 1):.1f}")
print()


def triples_at(rec, sid):
    if sid is None:
        return []
    for s in rec["sentences"]:
        if s["sentence_id"] == sid:
            return s["triples"]
    return []


yes_records = [r for r in records if r["contradiction"] == "YES"]
gold_matched = [r for r in yes_records if r["gold_evidence_sentence_id"] is not None and r["gold_ref_sentence_ids"]]
gold_usable = [
    r for r in gold_matched if triples_at(r, r["gold_evidence_sentence_id"]) and any(triples_at(r, sid) for sid in r["gold_ref_sentence_ids"])
]

print(f"YES docs: {len(yes_records)}")
print(f"  with both evidence + ref sentence_ids matched: {len(gold_matched)}")
print(f"  AND >= 1 triple at evidence and at >= 1 ref:   {len(gold_usable)}")
print()

if gold_usable:
    r = gold_usable[0]
    ev_t = triples_at(r, r["gold_evidence_sentence_id"])
    ref_t = [t for sid in r["gold_ref_sentence_ids"] for t in triples_at(r, sid)]
    print(f"Sample usable gold pair (doc_id={r['doc_id']}):")
    print(f"  evidence triples ({len(ev_t)}):")
    for t in ev_t:
        print(f"    {t}")
    print(f"  ref triples ({len(ref_t)}):")
    for t in ref_t:
        print(f"    {t}")

Documents extracted: 100 / 100
Total sentences:    3664
Total triples:      7129
Avg triples / doc:  71.3

YES docs: 50
  with both evidence + ref sentence_ids matched: 36
  AND >= 1 triple at evidence and at >= 1 ref:   36

Sample usable gold pair (doc_id=3499318673_1):
  evidence triples (1):
    {'s': 'Little Gidding', 'p': 'is', 'o': "the first poem of T. S. Eliot 's Four Quartets"}
  ref triples (2):
    {'s': 'Little Gidding', 'p': 'is', 'o': "the fourth and final poem of T. S. Eliot 's Four Quartets"}
    {'s': "T. S. Eliot 's Four Quartets", 'p': 'discuss', 'o': 'time, perspective, humanity, and salvation'}
